# Setup

## Imports

In [8]:
import math
import pickle
from dataclasses import dataclass
from pathlib import Path

import cv2
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.io as pio
import scipy.ndimage as ndimage
import tifffile
import xarray as xr
from scipy import interpolate, signal
from scipy.ndimage import shift
from scipy.stats import pearsonr, zscore
from skimage import morphology
from skimage.registration import phase_cross_correlation
from tqdm import tqdm

import my_functions as f
from my_general_variables import *

pio.templates.default = "plotly_dark"

pd.set_option("mode.copy_on_write", True)
pd.set_option("compute.use_numba", True)
pd.set_option("compute.use_numexpr", True)
pd.set_option("compute.use_bottleneck", True)

## Paths

In [9]:
path_home = Path(r'E:\2024 03_Delay 2-P multiple planes')


# fish_names = [folder.stem for folder in path_home.iterdir() if folder.is_dir()]
# fish_names.remove('Behavior')


# for fish_name in fish_names:
# fish_name = r'20240228_01_delay_2p-1_mitfaMinusMinus,elavl3H2BCaMP6s_7dpf'

fish_name = r'20240415_02_delay_2p-2_mitfaMinusMinus,elavl3H2BGCaMP6f_5dpf'
# '20240314_01_delay_2p-1_mitfaMinusMinus,elavl3H2BGCaMP6s_6dpf'
# "20240404_01_delay_2p-1_mitfaMinusMinus,elavl3H2BGCaMP6s_7dpf"
# '20240327_01_delay_2p-1_mitfaMinusMinus,elavl3H2BGCaMP6s_7dpf'

# '20240313_02_delay_2p-1_mitfaMinusMinus,elavl3H2BGCaMP6s_5dpf'
# '20240321_01_delay_2p-1_mitfaMinusMinus,elavl3H2BGCaMP6s_6dpf'
# '20240311_02_delay_2p-1_mitfaMinusMinus,elavl3H2BGCaMP6s_5dpf'

#! '20240304_01_delay_2p-1_mitfaMinusMinus,Gal4elavl3GCaMP6s_6dpf' is good


# behavior_path = Path(r'D:\2024 02_Delay 2p\Behavior')
# imaging_path = Path(r'D:\2024 02_Delay 2p') / fish_name / 'Imaging'

behavior_path = path_home / 'Behavior'
imaging_path = path_home / fish_name / 'Imaging'

protocol_path = behavior_path / (fish_name + '_stim control.txt')
camera_path = behavior_path / (fish_name + '_cam.txt')
tracking_path = behavior_path / (fish_name + '_mp tail tracking.txt')


galvo_path = imaging_path / 'signalsfeedback.xls'
images_path = imaging_path / (fish_name + '_green.tif')

anatomy_1_path = path_home / fish_name / 'Anatomical stack 1.tif'
anatomy_1_filtered_path = path_home / fish_name / 'Anatomical stack 1 binned and filtered.tif'

## Parameters

In [10]:
light_percentage_increase_thr = 5
average_light_derivative_thr = 10
top_bottom_frame_slice = 50  # number pixels
front_back_frame_slice = 50  # number pixels


# step_size = 0.001  # mm
# number_repetitions_the_plane = 20
images_bin_size = 30
number_repetitions_the_plane_consecutively_stable = 40
# step_between_repetitions_of_the_plane = 1



median_filter_kernel = 3

# kernel_size = 3
# ddepth = cv2.CV_16S


total_motion_thr = 0.5


#! debug
nrows = None
# 100000000

number_rows_read = None


galvo_value_height_threshold = 0.5
galvo_value_distance_threshold = 100
galvo_value_width_threshold = 20


xy_movement_allowed = 0.15  # fraction of the real image


number_imaged_planes = 15
number_reps_plane_consective = 2
relevant_cs = np.concatenate([range(5,35), range(45,75)])




motion_thr_from_trial_average = 5


#! this is overwriting the one in my_general_variables.py
# cols_to_use_orig = ['FrameID'] + ['x15'] + ['y15'] + ['angle15']
# data_cols = ['X 14'] + ['Y 14'] + ['Angle (deg) 14']
# angle_name = 'Angle (deg) 14'
# angle_cols = [angle_name]

time_experiment_f = frame_id

## Functions

In [11]:
def get_bytes_header_and_image(images_path):
	
	byte_number = 0

	# byte_order = np.fromfile(images_path, dtype=np.uint16, count=1, offset=byte_number)[0]
	byte_number += 2

	# arbitrary = np.fromfile(images_path, dtype=np.uint16, count=1, offset=byte_number)[0]
	byte_number += 2

	# IGD1off = np.fromfile(images_path, dtype=np.uint32, count=1, offset=byte_number)[0]
	byte_number += 4

	number_fields = np.fromfile(images_path, dtype=np.uint16, count=1, offset=byte_number)[0].byteswap()
	byte_number += 2

	# tag = np.fromfile(images_path, dtype=np.uint16, count=1, offset=byte_number)[0]
	byte_number += 2

	# field_data = np.fromfile(images_path, dtype=np.uint8, count=6, offset=byte_number)
	byte_number += 6

	width = np.fromfile(images_path, dtype=np.uint32, count=1, offset=byte_number)[0].byteswap()
	byte_number += 4	

	# tag = np.fromfile(images_path, dtype=np.uint16, count=1, offset=byte_number)[0]
	byte_number += 2

	# field_data = np.fromfile(images_path, dtype=np.uint8, count=6, offset=byte_number)
	byte_number += 6

	height = np.fromfile(images_path, dtype=np.uint32, count=1, offset=byte_number)[0].byteswap()
	byte_number += 4

	for n in range(number_fields - 2):
		# np.fromfile(images_path, dtype=np.uint8, count=12, offset=byte_number)
		byte_number += 12

	# next_offset = np.fromfile(images_path, dtype=np.uint32, count=1, offset=byte_number)[0].byteswap()
	byte_number += 4

	# skip resolution data
	# np.fromfile(images_path, dtype=np.uint8, count=16, offset=byte_number)
	byte_number += 16

	# number_pixels = height * width

	# All bytes up to here subatracted by the 2 bytes used to store the byte order.
	bytes_header = number_fields*12+2+4+16

	# Because images are uint16.
	# bytes_image = number_pixels * 2

	# print('bytes_header', bytes_header)

	return bytes_header, height, width

def get_image_from_tiff(images_path, image_i, bytes_header, height, width):

	# 2 comes from the 2 bytes used to store the byte order.
	offset_image = 2 + (image_i) * (bytes_header + height * width * 2) + bytes_header

	image_data = np.fromfile(images_path, dtype=np.uint16, count=height*width, offset=offset_image).byteswap().reshape((height, width))

	return image_data

def get_number_images(images_path, bytes_header_and_image):

	total_tif_size = os.path.getsize(images_path)

	number_images = (total_tif_size - 2) // bytes_header_and_image

	return number_images




def find_plane_in_anatomical_stack(anatomical_stack_images, the_plane_mean_subset_last_images, plane_where_we_are, x_dim, y_dim):

	#* Handle to the multipage TIFF file with the plane being imaged.
	# the_plane_tiff = tifffile.TiffFile(the_plane_path)

#! explain
	# the_plane_mean_of_last_images = the_plane_tiff.asarray(slice(-number_repetitions_of_the_plane_to_analyze-1,-1,step_between_repetitions_of_the_plane_to_analyze))
	the_plane_mean_subset_last_images = the_plane_mean_subset_last_images[y_dim:-y_dim, x_dim:-x_dim]
	
	if plane_where_we_are is not None:
		
		if (first_plane_substack := plane_where_we_are - number_planes_around_the_plane) < 0:
			
			first_plane_substack = 0

		if (last_plane_substack := plane_where_we_are + number_planes_around_the_plane + 1) > len(anatomical_stack_images):
			
			last_plane_substack = len(anatomical_stack_images)

		anatomical_stack_images_ = anatomical_stack_images[first_plane_substack : last_plane_substack]
		
	else:
		
		anatomical_stack_images_ = anatomical_stack_images

		first_plane_substack = 0
		
	template_matching_results = [cv2.matchTemplate(plane, the_plane_mean_subset_last_images, cv2.TM_CCOEFF_NORMED) for plane in anatomical_stack_images_]

	# b = [np.array(x.flatten())[np.argpartition(x.flatten(), 3)[:3]] for x in template_matching_results]

	# # [print(ii) for ii in b]

	# a = np.mean(b, axis=1)

	plane_i = np.argmax([x.max() for x in template_matching_results])
	# np.argmax(a)

	xy_in_plane = np.argmax(template_matching_results[plane_i][0]), np.argmax(template_matching_results[plane_i][1])
	
	plane_i += first_plane_substack

	# Find the index of the maximum correlation value
	return plane_i, xy_in_plane


def read_camera(camera_path):

	try:
		# start = timer()
		
		# camera = pd.read_csv(str(camera_path), sep='\t', header=0, decimal='.', skiprows=[*range(1,number_frames_discard_beg)])
		camera = pd.read_csv(camera_path, engine='pyarrow', sep=' ', header=0, decimal='.')
		# , na_filter=False
		# dtype={time_experiment_f : 'int64', abs_time : 'int64', ela_time : 'float64'})
		# skipfooter=1
		camera = camera.iloc[:-1,:]

		camera.rename(columns={'FrameID' : frame_id, 'ID' : frame_id}, inplace=True)
		
		# print('Time to read cam.txt: {} (s)'.format(timer()-start))
		
		return camera

	except:

		print('Issues in the camera log file')
		
		return None

def framerate_and_reference_frame(camera):

	# first_frame_absolute_time = camera[abs_time].iloc[0]

	camera = camera.drop(columns=abs_time, errors='ignore')

	camera_diff = camera[ela_time].diff()

	print('Max IFI: {} ms'.format(camera_diff.max()))
	
	# First estimate of the interframe interval, using the median
	ifi = camera_diff.median()
	# camera_diff.iloc[number_frames_discard_beg : ].median()
	print('First estimate of IFI: {} ms'.format(ifi))


	camera_diff_index_correct_IFI = np.where(abs(camera_diff - ifi) <= max_interval_between_frames)[0]

	camera_diff_index_correct_IFI_diff = np.diff(camera_diff_index_correct_IFI)

	reference_frame_id = 0
	last_frame_id = 0

	#* Find a region at the beginning where the IFI from frame to frame does not vary significantly and is similar to the first estimate of the true IFI (ifi).
	for i in range(1, len(camera_diff_index_correct_IFI_diff)):

		if camera_diff_index_correct_IFI_diff[i-1] == 1 and camera_diff_index_correct_IFI_diff[i] == 1:

			reference_frame_id = camera[frame_id].iloc[camera_diff_index_correct_IFI[i] - 1]


			# # first_frame_absolute_time is not None when there is absolute time in the cam file.
			# if first_frame_absolute_time is not None:
			# 	reference_frame_time = first_frame_absolute_time + camera[ela_time].iloc[camera_diff_index_correct_IFI[i] - 1] - camera[ela_time].iloc[0]
			# else:
			# 	reference_frame_time = None

			break

	#* Find a similar region but at the end of the experiment.
	for i in range(len(camera_diff_index_correct_IFI_diff)-1, 0, -1):

		if camera_diff_index_correct_IFI_diff[i-1] == 1 and camera_diff_index_correct_IFI_diff[i] == 1:
			
			last_frame_id = camera[frame_id].iloc[camera_diff_index_correct_IFI[i] - 1]
			#last_frame_time = first_frame_absolute_time + camera[time].iloc[camera_diff_index_right_IFI[i] - 1] - camera[time].iloc[0]

			break


	#* Second estimate of the interframe interval, using the mean, and assuming there is no increasing accumulation of frames in the buffer during the experiment; Only the region between the two frames identified in the previous two for loops is considered.
	ifi = camera_diff.iloc[reference_frame_id - camera[frame_id].iloc[0] : last_frame_id - camera[frame_id].iloc[0]].mean()

	print('Second estimate of IFI: {} ms'.format(ifi))
	predicted_framerate = 1000 / ifi
	print('Estimated framerate: {} FPS'.format(predicted_framerate))


	# Lost_frames = lost_frames(camera, camera_diff, ifi, fish_name, fig_camera_name)

	return predicted_framerate, reference_frame_id

def read_tail_tracking_data(data_path):

	# Angles in data come in radians.

	try:
		
		# start = timer()
		
		data = pd.read_csv(data_path, engine='pyarrow', sep=' ', usecols=cols_to_use_orig, header=0, decimal='.', na_filter=False, names=[frame_id]+data_cols)
		# dtype=dict(zip(cols_to_use_orig, ['int64'] + ['float32']*len(cols_to_use_orig))))
		# skipfooter=1
		data = data.iloc[:-1,:]
		
		#* Right now, pyarrow engine ignores renaming when opening the csv.
		data.rename(columns=dict(zip(cols_to_use_orig, [frame_id] + data_cols)), inplace=True)

		# print('Time to read tail tracking .txt: {} (s)'.format(timer()-start))


		#? maybe before this was necessary because "decimal" in pd.read_csv was set to ",".
		#* Even if decimal separator is wrong, this will correct it.
		# data.iloc[:,1:] = data.iloc[:,1:].astype('float32')

		#* Convert tail tracking data from radian to degree
		data.loc[:,angle_cols] *= (180/np.pi)
		
		return data

	except:
		
		#TODO
		# f.save_info(protocol_info_path, self.metadata.name, 'Tail tracking might be corrupted!')
		print('Issues in tail tracking data')
		return None

def plot_behavior_overview(data, fish_name, fig_behavior_name):
	# data containing tail_angle.

	# mask_frames = np.ones(number_frames + round(60*framerate), dtype=bool)
	# mask_frames[:: round(framerate * 0.5)] = False
	# mask_frames[0] = False
	
	# rows_to_skip = np.arange(number_frames + round(60*framerate))
	# rows_to_skip = rows_to_skip[mask_frames]

	# start = timer()

	# overall_data = pd.read_csv(data, sep=' ', header=0, usecols=cols, skiprows=rows_to_skip, decimal=',')
	# overall_data = overall_data.astype('float32')

	# print(timer() - start)
	plt.figure(figsize=(30, 15))
	plt.plot(data[frame_id]/expected_framerate/60/60, data[tail_angle], 'black')
	plt.xlabel('Time (h)')
	plt.ylabel('Tail end angle (deg)')
	plt.suptitle('Behavior overview\n' + fish_name)
	# plt.show()
	# plt.legend(frameon=False, loc='upper center', ncol=2)
	plt.savefig(fig_behavior_name, dpi=100, bbox_inches='tight')
	plt.close()

def tracking_errors(data, single_point_tracking_error_thr = single_point_tracking_error_thr):

	if ((a := data.loc[:, angle_cols].abs().max()) > single_point_tracking_error_thr).any():
		print('Possible tracking error; max(abs(angle of individual point)): ')
		print(a)

		return True

	elif data.iloc[:,1:].isna().to_numpy().any():
		print('Possible tracking failures; there are NAs in data')

		return True

	else:
		return False

def merge_camera_with_data(data, camera):

	data = pd.merge_ordered(data, camera, on=frame_id, how='outer')

	data[frame_id] -= data[frame_id].iat[0]

	return data

def interpolate_data(data, predicted_framerate, expected_framerate=expected_framerate):
	# expected_framerate is the framerate to which data is interpolated. So, output data is as if it had been acquired at the expected_framerate (700 FPS when I wrote this).

	data_ = data.copy()

	#* Interpolate tail tracking data to the expected framerate.

	data_[frame_id] *= expected_framerate/predicted_framerate

	data_.rename(columns={frame_id : time_experiment_f}, inplace=True)

	interp_function = interpolate.interp1d(data_[time_experiment_f], data_.drop(columns=time_experiment_f), kind='slinear', axis=0, assume_sorted=True, bounds_error=False, fill_value="extrapolate")

	data = pd.DataFrame(np.arange(data_[time_experiment_f].iat[0], data_[time_experiment_f].iat[-1]), columns=[time_experiment_f])

	data[data_.drop(columns=time_experiment_f).columns] = interp_function(data[time_experiment_f])

	return data

def read_protocol(protocol_path):

	#* Read protocol file.
	if Path(protocol_path).exists():
		# protocol = pd.read_csv(str(protocol_path), sep=' ', header=0, names=['Type', beg, end], usecols=[0, 1, 2], index_col=0)
		protocol = pd.read_csv(protocol_path, engine='pyarrow', sep=' ', header=0, decimal='.', names=['Type', beg, end])
		# dtype={'Type' : 'str', beg : 'int', end : 'int'})

	else:
		print('The stim log file does not exist')
		return None

	#* Were the stimuli timings not saved?
	if protocol.empty:
		print('The stim log file is empty')
		return None

	if protocol.iloc[0,0] == 0:
		print('The stim log file is currupted')
		return None

	#* Right now, pyarrow engine ignores renaming when opening the csv.
	protocol.rename(columns={'Beg' : beg, 'End' : end}, inplace=True)
	protocol['Type'] = protocol['Type'].replace({'Cycle' : cs, 'Reinforcer' : us})
	protocol.sort_values(by=beg, inplace=True)
	protocol = protocol.set_index('Type')

	return protocol

def lost_stim(number_cycles, number_reinforcers, expected_number_cs_trials, expected_number_us_trials):

	if number_cycles < expected_number_cs_trials:

		# save_info(protocol_info_path, fish_name, 'Not all CS! Stopped at CS {} ({}).'.
		# format(number_cycles, id_debug))
		print('Not all CS!')

	if number_reinforcers < expected_number_us_trials:
		
		# save_info(protocol_info_path, fish_name, 'Not all US! Stopped at US {} ({}).'.format(number_reinforcers, id_debug))
		print('Not all US!')
			
	if number_cycles < expected_number_cs_trials or number_reinforcers < expected_number_us_trials:

		return True
	
	else:
		return False

def protocol_info(protocol):

	#* Count the number of cycles, trials, blocks and bouts.
	# Using len() just in case these is a single element.
	# number_cs = len(protocol.loc['Cycle', beg])

	if protocol.index.isin([us]).any():
		# number_us = len(protocol.loc[us, beg])

		us_beg = protocol.loc[us, beg]
		us_end = protocol.loc[us, end]
		us_dur = (us_end - us_beg).to_numpy() # in ms
		us_isi = (us_beg[1:] - us_end[:-1]).to_numpy() / 1000 / 60 # min
	else:
		# number_us = 0

		us_dur = None
		us_isi = None

	# habituation_duration = protocol.iloc[0,0] / 1000 / 60 # min

	cs_beg = protocol.loc[cs, beg]
	cs_end = protocol.loc[cs, end]
	cs_dur = (cs_end - cs_beg).to_numpy() # in ms
	cs_isi = (cs_beg[1:] - cs_end[:-1]).to_numpy() / 1000 / 60 # min


	return cs_dur, cs_isi, us_dur, us_isi

def plot_protocol(cs_dur, cs_isi, us_dur, us_isi, fish_name, fig_protocol_name):

	plt.figure(figsize=(14,14))
	plt.plot(np.arange(1, len(cs_isi) + 1), cs_isi, label='inter-cs interval\nmin int.=' + str(round(np.amin(cs_isi)*60,1)) + ' s\n' + 'cs min dur=' + str(round(np.amin(cs_dur)/1000,3)) + ' s\n' + 'cs max dur=' + str(round(np.amax(cs_dur)/1000,3)) + ' s')
	
	plt.plot(np.arange(5, 4+len(us_isi)+1), us_isi, label='inter-us interval\nmin int.=' + str(round(np.amin(us_isi)*60,1)) + ' s\n' + 'us min dur=' + str(round(np.amin(us_dur)/1000,3)) + 's\n' + 'us max dur='+ str(round(np.amax(us_dur)/1000,3)) + ' s')
	plt.xlabel('Trial number')
	plt.ylabel('ISI (min)')
	plt.ylim(0, 10)
	plt.legend(frameon=False, loc='upper center', ncol=2)
	plt.suptitle('Summary of protocol\n' + fish_name)
	plt.savefig(fig_protocol_name, dpi=100, bbox_inches='tight')
	plt.close()

def identify_trials(data, protocol):

	data[[cs, us]] = [0, 0]

	for cs_us in [cs, us]:

		protocol_sub = protocol.loc[cs_us, [beg, end]].to_numpy()

		for i, p in enumerate(protocol_sub):
			
			data.loc[data[abs_time].between(p[0], p[1]), cs_us] = i + 1

	data = data.set_index(abs_time)

	try:
		data.loc[:, data_cols] = data.loc[:, data_cols].interpolate(kind='slinear')
	except:
		print('HERE. FIX THIS')

	#! data = data.reset_index(drop=True).dropna()
	data = data.reset_index().dropna()

	data[time_experiment_f] = data[time_experiment_f].astype('int64')

	data[[cs, us]] = data[[cs, us]].astype('Sparse[int16]')

	#* Fix dtypes.
	# for cs_us in [cs, us]:

	# 	data[cs_us] = data.loc[:, cs_us].astype(pd.CategoricalDtype(categories=data[cs_us].unique(), ordered=True))

	return data




def get_good_images_indices(images_subset):


	top = np.nanmean(images_subset[:, :top_bottom_frame_slice, :], axis=(1,2))
	bottom = np.nanmean(images_subset[:, -top_bottom_frame_slice:, :], axis=(1,2))
	front = np.nanmean(images_subset[:, :, -front_back_frame_slice:], axis=(1,2))
	back = np.nanmean(images_subset[:, :, :front_back_frame_slice], axis=(1,2))

	all = np.nanmean(images_subset, axis=(1,2))
	all_median = np.nanmedian(all)

	light_percentage_change = (np.abs(all - all_median) / all_median) * 100

	#* Discard based on overall light (too low or too high)
	mask_good_images = light_percentage_change < light_percentage_increase_thr

	#* And also discard based on the derivative
	mask_good_images = mask_good_images & ([True] + list((np.abs(np.diff(top)) < average_light_derivative_thr) & (np.abs(np.diff(bottom)) < average_light_derivative_thr) & (np.abs(np.diff(front)) < average_light_derivative_thr) & (np.abs(np.diff(back)) < average_light_derivative_thr) & (np.abs(np.diff(all) < average_light_derivative_thr))))

	# plt.plot(top-np.nanmedian(top))
	# plt.plot(bottom-np.nanmedian(bottom))
	# plt.plot(front-np.nanmedian(front))
	# plt.plot(back-np.nanmedian(back))
	# plt.plot(all-np.nanmedian(all))
	# plt.plot(light_percentage_change)
	# plt.plot(np.where(mask_good_images, mask_good_images, np.nan)*(-10), lw=3)
	# plt.legend(['Top', 'Bottom', 'Front', 'Back', 'Whole', r'Whole % change', r'Good images'], loc='center left', bbox_to_anchor=(1, 0.5))
	# plt.show()
	
	return mask_good_images


def get_fixed_number_good_last_images(images_subset):

	mask_good_images = get_good_images_indices(images_subset)

	#* Discard the first and last frames
	mask_good_images[:3] = False
	mask_good_images[-3:] = False

	#* Find consecutive True regions
	new_mask = np.zeros_like(mask_good_images, dtype=bool)
	consecutive_count = 0
	for i, value in enumerate(mask_good_images[::-1]):

		if value:
			consecutive_count += 1
			if consecutive_count >= number_repetitions_the_plane_consecutively_stable:
				new_mask[i - number_repetitions_the_plane_consecutively_stable + 1 : i + 1] = True
				break
		else:
			consecutive_count = 0
	mask_good_images = new_mask[::-1]
	

	images_subset = images_subset[mask_good_images,:,:]

	# plt.plot(mask_good_images)
	# # plt.axvspan(np.diff(mask_good_images)==1, np.diff(mask_good_images)==-1)
	# plt.legend([r'Good images'], loc='center left', bbox_to_anchor=(1, 0.5))
	# plt.show()

	return images_subset


def get_maximum_number_good_last_images(images_subset):

	mask_good_images = get_good_images_indices(images_subset)

	#* Discard the first and last frames
	mask_good_images[:3] = False
	mask_good_images[-3:] = False

	#* Find consecutive True regions
	new_mask = np.zeros_like(mask_good_images, dtype=bool)
	consecutive_count = 0
	for i, value in enumerate(mask_good_images[::-1]):

		if value:
			consecutive_count += 1
			# if consecutive_count >= number_repetitions_the_plane_consecutively_stable:
			# 	new_mask[i - number_repetitions_the_plane_consecutively_stable + 1 : i + 1] = True
			# 	break
		else:
			if consecutive_count > 0 and consecutive_count > number_repetitions_the_plane_consecutively_stable:
				new_mask[i - consecutive_count : i] = True
				break
			else:
				consecutive_count = 0
			
	mask_good_images = new_mask[::-1]
	

	images_subset = images_subset[mask_good_images,:,:]

	# plt.plot(mask_good_images)
	# # plt.axvspan(np.diff(mask_good_images)==1, np.diff(mask_good_images)==-1)
	# plt.legend([r'Good images'], loc='center left', bbox_to_anchor=(1, 0.5))
	# plt.show()

	return images_subset


def get_template_image(frames):

	template_image = ndimage.median_filter(np.nanmean(frames, axis=0), size=median_filter_kernel)
	# np.mean(ndimage.median_filter(frames, size=median_filter_kernel, axes=(1,2)), axis=0)

	# plt.figure(figsize=(20, 16))
	# plt.imshow(template_image)
	# plt.colorbar(shrink=0.5)
	# plt.title('Anatomy')
	# plt.show()

	return template_image



def measure_motion(frames, anatomy, normalization=None):

	x_motion=np.zeros(np.shape(frames)[0])
	y_motion=np.zeros(np.shape(frames)[0])
	for j in range(frames.shape[0]):
		X=phase_cross_correlation(anatomy, frames[j,:,:], upsample_factor=10, space='real', normalization=normalization, overlap_ratio=0.9)
		x_motion[j]=X[0][0]
		y_motion[j]=X[0][1]

	return np.column_stack([x_motion, y_motion])


def get_total_motion(motion):
	# total_motion=np.zeros(np.shape(frames)[0])
	total_motion = np.linalg.norm(motion, axis=1)

	# plt.show()
	# fig, axs = plt.subplots(1, 2)
	# fig.suptitle('Motion of each frame')
	# axs[0].plot(total_motion, 'k.')
	# axs[1].scatter(motion[:,0]-0.01+0.02*np.random.rand(motion[:,0].shape[0]),motion[:,1]-0.01+0.02*np.random.rand(motion[:,1].shape[0]),s=0.5)
	# # fig.show()
	# plt.show()

	return total_motion



def align_frames(frames, motion, total_motion, total_motion_thr=None):

	# total_motion = get_total_motion(motion)

	##* Discard frames with too much motion.
	if total_motion_thr is not None:
		frames_indices_ignore = np.where(total_motion > total_motion_thr)[0]
	else:
		frames_indices_ignore = []
	
	aligned_frames=np.zeros(frames.shape)

	for j in range(frames.shape[0]):
		if j not in frames_indices_ignore:
			aligned_frames[j,:,:]=shift(frames[j,:,:], motion[j], output=None, order=3, mode='constant', cval=0.0, prefilter=True)
		#   the commented lines below check that the shift was performed in the correct direction	
			# X=phase_cross_correlation(original_anatomy, frames[j,:,:] ,upsample_factor=10, space='real')
			# print(X)
			# Y=phase_cross_correlation(original_anatomy, aligned_frames[j,:,:] ,upsample_factor=10, space='real')
			# print(Y)
   
	return aligned_frames


def correct_motion_within_trial(trial, number_iterations=5):

	images_trial_ = trial.images.to_numpy()

	template_image_ = get_template_image(get_maximum_number_good_last_images(images_trial_))

	motion_thr = 5

	for _ in range(number_iterations):

		motion_thr = int(np.ceil(motion_thr))
		motion_thr = motion_thr if motion_thr > 5 else 5
		
		#* Measure motion of each frame using phase cross-correlation.
		motion = measure_motion(images_trial_[:, motion_thr:-motion_thr, motion_thr:-motion_thr], template_image_[motion_thr:-motion_thr, motion_thr:-motion_thr], normalization=None)

		#* Measure motion of each frame using phase cross-correlation.
		total_motion = get_total_motion(motion)
		# Use half of the frames to get the template image.
		motion_thr = np.median(total_motion)

		#* Align the frames to their average.
		aligned_frames = align_frames(images_trial_, motion, total_motion, 5)
		
		#* Motion correction relative to trials average.
		template_image_ = get_template_image(aligned_frames[np.where(total_motion <= motion_thr)[0]])


	plt.imshow(ndimage.median_filter(np.mean(aligned_frames, axis=0), size=median_filter_kernel))
	# plt.imshow(np.mean(ndimage.median_filter(aligned_frames, size=median_filter_kernel, axes=(1,2)), axis=0))

	#* Identify the plane number of the trial.
	plane_number, _ = find_plane_in_anatomical_stack(anatomical_stack_images, template_image_.astype('float32'), None, x_dim, y_dim)

	return motion, template_image_, plane_number



def correct_motion_across_trials():

	return

## Classes

In [12]:
@dataclass
class Trial:
	
	trial_number : int

	# position_anatomical_stack : int
	# reference_image : np.ndarray

	protocol : pd.DataFrame
	behavior : pd.DataFrame
	images : xr.DataArray


@dataclass
class Plane:

	trials : list[Trial]

	# reference_image_position_anatomical_stack : int
	# reference_image : np.ndarray

	# order_planes_sequence : int

	def get_reference_position(self):

		return round(np.median([trial.position_anatomical_stack for trial in self.trials]))

@dataclass
class Data:
	
	planes : list[Plane]



# Analysis

## Align all data

In [ ]:
#! flag summer time
if (date := int(fish_name.split('_')[0][4:6])) >= 4 and date <= 10:
	Summer_time = True
else:
	Summer_time = False




#region Behavior camera

data = read_camera(camera_path)
data[abs_time] = data[abs_time].astype('float64')

print('Behavior camera started: ', pd.to_datetime(data[abs_time].iat[0], unit='ms'))
 

#* Estimate the true framerate.
predicted_framerate, reference_frame_id = framerate_and_reference_frame(data)


data = data.drop(columns=ela_time)

#* Discard frames that will not be used (in camera and hence further down).
# The calculated interframe interval before the reference frame is variable. Discard what happens up to then (also achieved by using how='inner' in merge_camera_with_data).
data = data[data[frame_id] >= reference_frame_id]

		# #! reverse data_cols to what we want
		# #! #* Open tail tracking data.
		# data = read_tail_tracking_data(tracking_path)

		# # if (data := read_tail_tracking_data(data_path)) is None: # type: ignore
		# # 	return None

		# # plot_behavior_overview(data, fish_name, fig_behavior_name)

		# # #* Look for possible tail tracking errors.
		# # if tracking_errors(data, single_point_tracking_error_thr):
		# # 	return None

		# #! #* Add information about the time of each frame to data.
		# data = merge_camera_with_data(data, camera)
		# # data = camera

#* Fix abs_time so that the time of each frame becomes closer to the time at which the frames were acquired by the data and not when they were caught by the computer.
# The delay between acquiring and catching the frame is unknown and therefore disregarded.
data[abs_time] = np.linspace(data[abs_time].iat[0], data[abs_time].iat[0] + len(data) * (1000 / predicted_framerate), len(data))

#! Need to join with galvo before doing this.
#! #* Interpolate data to the expected framerate.
# data = interpolate_data(data, predicted_framerate)

#endregion




#region Stim log and merge it with the behavior camera data

#* Open the stim log.
protocol = read_protocol(protocol_path)


# protocol.iloc[:,1] - protocol.iloc[:,0]

#* Identify the stimuli, trials of the experiment.
data_cols = []
data = identify_trials(data, protocol)

# plt.plot(data[abs_time])
# data[cs].unique()
#endregion

# AA = data.loc[data[cs] == 10, :]
# (data.loc[data[cs] == 10, abs_time].iloc[-1] - data.loc[data[cs] == 10, abs_time].iloc[0])

# plt.plot(AA[abs_time], AA[cs])

# AA[cs].min()



#region Galvo signal
#* Read the galvo signal.
galvo = pd.read_csv(galvo_path, sep='\t', decimal=',', usecols=[0,1], names=[abs_time, 'GalvoValue'], dtype={'GalvoValue':'float64'}, parse_dates=[abs_time], date_format=r'%d/%m/%Y  %H:%M:%S,%f', skip_blank_lines=True, skipinitialspace=True, nrows=nrows).dropna(axis=0)
galvo = galvo.reset_index(drop=True)
#* Convert the time in galvo to unixtime in ms
galvo[abs_time] = galvo[abs_time].astype('int64') / 10**6

#* To align the galvo signals to the respective frames with need to consider at least from the beginning of the galvo signal.
# if (first_timepoint_galvo := galvo[abs_time].iat[0]) <= (first_timepoint_data := data[abs_time].iat[0]):

# 	galvo = galvo[galvo[abs_time] >= first_timepoint_data]
# 	first_timepoint_galvo = galvo[abs_time].iat[0]

# plt.plot(galvo[abs_time])



print('Galvo started: ', pd.to_datetime(galvo[abs_time].iat[0], unit='ms'))

if Summer_time:
	galvo[abs_time] -= 60*60*1000




#* Remove consecutive duplicates...
galvo = galvo[galvo[galvo_value].ne(galvo[galvo_value].shift())]


galvo = galvo.reset_index(drop=True)



#* Find the beginning of the frames.
beg_first_image = signal.find_peaks(galvo[galvo_value], height=0.5, prominence=0.05)[0][0]
beg_first_image_time = galvo[abs_time].iat[beg_first_image]

peaks = signal.find_peaks(galvo[galvo_value].iloc[beg_first_image+1000:], height=[0.5, 5], distance=100, prominence=[0.5, 5], width=20)[0] + beg_first_image + 1000
number_peaks = len(peaks)

galvo_sub = galvo.loc[0:10000]

fig, ax = plt.subplots()
ax.set_title('')
ax.plot(galvo_sub[abs_time].to_numpy(), galvo_sub[galvo_value].to_numpy(), 'k')
ax.plot(beg_first_image_time, 1, 'ro')
ax.plot(galvo_sub[abs_time].iloc[peaks[:5]], galvo_sub[galvo_value].iloc[peaks[:5]], 'bo')
ax.set_xlabel('Interframe interval (ms)')
ax.set_ylabel('Galvo value')
fig.show()



#* Calculate the interframe interval.
interframe_interval_array = galvo[abs_time].iloc[peaks].diff()
interframe_interval = interframe_interval_array.median()


print('The median of the interframe interval is:', interframe_interval)
print('Min and max interframe interval:', interframe_interval_array.min(), interframe_interval_array.max())






#* Discard a few images at the beginning where the imaging is not good or we are unsure of the true beginning of the images.
beg_image_to_consider_index = interframe_interval_array.index[np.where(interframe_interval_array == interframe_interval)[0][0] - 1]
beg_image_to_consider_time = galvo[abs_time].iat[beg_image_to_consider_index]

peaks = peaks[peaks >= beg_image_to_consider_index]


#* Number of images to discard at the beginning.

# need to round donw
number_images_before_first_image_to_consider = round((beg_image_to_consider_time - beg_first_image_time) / interframe_interval)
# beg_first_image_time = beg_image_to_consider_time - number_images_before_first_image_to_consider * interframe_interval




#* Get info about the images in the multipage tiff with the imaging data.
bytes_header, height, width = get_bytes_header_and_image(images_path)
bytes_header_and_image = bytes_header + height * width * 2

#* Find where we started imaging in the anatomical stack.
number_images = get_number_images(images_path, bytes_header_and_image)
# - number_images_before_first_image_to_consider


print('Number of galvo peaks identified:', number_peaks)
print('Number of images in the tiff:', number_images)


# Here I use NAN and not False to later be able to discard rows where a frame did not start and where there is no tracking data.
galvo['Frame beg'] = np.nan
galvo.loc[galvo.iloc[peaks].index[:number_images-number_images_before_first_image_to_consider], 'Frame beg'] = 1
# np.arange(1, len(peaks)+1)

galvo_sub = galvo.iloc[0:5000].copy()

#* Add the beginning of the missed images to galvo.
galvo = pd.merge_ordered(galvo, pd.DataFrame({abs_time: np.arange(beg_image_to_consider_time - number_images_before_first_image_to_consider * interframe_interval, beg_image_to_consider_time, interframe_interval), 'Frame beg': np.ones(number_images_before_first_image_to_consider)}))



fig, ax = plt.subplots()
ax.plot(galvo_sub[abs_time].to_numpy(), galvo_sub[galvo_value].to_numpy(), 'k')
ax.plot(galvo_sub[abs_time].to_numpy(), galvo_sub['Frame beg'].to_numpy(), 'ro')
ax.plot(np.arange(beg_image_to_consider_time - number_images_before_first_image_to_consider * interframe_interval, beg_image_to_consider_time, interframe_interval), np.ones(number_images_before_first_image_to_consider)*2, 'yo')
ax.plot(galvo.iloc[0:5000][abs_time].to_numpy(), galvo.iloc[0:5000]['Frame beg'].to_numpy()*3, 'bo')
ax.set_xlabel('Interframe interval (ms)')
ax.set_ylabel('Galvo value')



galvo_sub = galvo.iloc[-5000:]

galvo_sub.loc[galvo_sub['Frame beg'].notna(), 'Frame beg'] = 1

fig, ax = plt.subplots()
ax.plot(galvo_sub[abs_time].to_numpy(), galvo_sub[galvo_value].to_numpy(), 'k')
ax.plot(galvo_sub[abs_time].to_numpy(), galvo_sub['Frame beg'].to_numpy(), 'ro')
ax.set_xlabel('Interframe interval (ms)')
ax.set_ylabel('Galvo value')
fig.show()

del galvo_sub

fig, ax = plt.subplots()
ax.plot(interframe_interval_array, 'k.')
ax.set_xlabel('Interframe interval (ms)')
ax.set_ylabel('Galvo value')
fig.show()

#endregion








#region Image data


#endregion


#region Merge galvo signal, stim log and behavior camera data


#* Discard imaging before the tracking started (in some cases, the tracking might start after the imaging).
if (first_timepoint_galvo := galvo[abs_time].iat[0]) <= (first_timepoint_data := data[abs_time].iat[0]):

	#* Do not discard images because only the galvo signal txt file is not overwritten when one stops and restarts the imaging.
	# number_images_discard = len(galvo[(galvo[abs_time] < first_timepoint_data) & (galvo['Frame beg'].notna())])
	# images = images[number_images_discard:]


	galvo = galvo[galvo[abs_time] >= first_timepoint_data]
	# first_timepoint_galvo = galvo[abs_time].iat[0]

	first_timepoint = galvo[abs_time].iat[0]

	data[abs_time] -= first_timepoint
	galvo[abs_time] -= first_timepoint

	data = data[data[abs_time] >= 0]
	galvo = galvo[galvo[abs_time] <= data[abs_time].iat[-1]]

	print('Galvo signal started before the tracking.')

	plt.plot(galvo[abs_time])
	plt.plot(data[abs_time])



first_timepoint = galvo[abs_time].iat[0]

data[abs_time] -= first_timepoint
galvo[abs_time] -= first_timepoint

data = data[data[abs_time] >= 0]
galvo = galvo[galvo[abs_time] <= data[abs_time].iat[-1]]








#* Update the number of images.
# Discard all images after the tracking is over.
number_images = len(galvo.loc[galvo['Frame beg'].notna(),:])


#* Get the images and align them to data.
#! Do not forget to discard the first images.
#!!!!! images = np.array([get_image_from_tiff(images_path, image_i, bytes_header, height, width) for image_i in range(number_images_before_first_image_to_consider, len(peaks))])
# images_subset_mean = [np.mean(image[-30:-10][-30:-10]).astype('float32') for image in images]
images = np.array([get_image_from_tiff(images_path, image_i, bytes_header, height, width).astype('float32') for image_i in tqdm(range(number_images))])

images.shape

images_mean = [image.mean() for image in images]







#* Merge galvo with data.
# data = pd.merge_ordered(data, galvo.drop(columns=galvo_value), on=abs_time, how='outer')
data = pd.merge_ordered(data, galvo, on=abs_time, how='outer')

del galvo, protocol


# Remove all colummns where there is no tracking data and no frame started.
# Really need to convert to dense...
data[[cs,us]] = data[[cs,us]].sparse.to_dense()
data = data.dropna(subset=[frame_id, 'Frame beg', cs, us], how='all')
# data.loc[data['Frame beg'].notna()]
# data[[frame_id, 'Frame beg']] = data[[frame_id, 'Frame beg']].fillna(0)


# data[abs_time] -= data[abs_time].iat[0]




# first_timepoint = galvo[abs_time].iat[0]

# galvo[abs_time] -= first_timepoint


# data[abs_time] -= first_timepoint
# data = data[data[abs_time] >= 0]


# galvo = galvo[galvo[abs_time] <= data[abs_time].iat[-1]]




# fig, axs = plt.subplots(2, 1, figsize=(10, 8))
# axs[0].plot(galvo[abs_time])
# axs[0].set_title('Galvo Signal')
# axs[1].plot(data[abs_time])
# axs[1].set_title('Data')
# plt.tight_layout()
# plt.show()

# plt.plot(data[abs_time].to_numpy())
# plt.plot(galvo[abs_time].to_numpy())



# data[abs_time] -= data[abs_time].iat[0]

# del galvo
# data = pd.merge_ordered(data, galvo, on=abs_time, how='outer')




data['Image mean'] = np.nan




data.loc[data['Frame beg'].notna(), 'Image mean'] = images_mean[:len(data.loc[data['Frame beg'].notna(), 'Image mean'])]
# data.loc[data['Frame beg'].notna(), 'Image mean'] = images_subset_mean_top[:len(data[data['Frame beg'].notna()])]

#endregion

## Check that protocol and images are aligned properly

In [ ]:
#region Check whether the different pieces of data are aligned.

data = data.reset_index(drop=True)


data[[cs, us]] = data[[cs, us]].fillna(0)

# data[cs].cat.remove_unused_categories()
# data[us].cat.remove_unused_categories()


x0 = 2500
x1 = 2500

stim_numbers = data.loc[data[us] != 0, us].unique()


fig, axs = plt.subplots(stim_numbers.size-1, 1, figsize=(10, 50), sharex=True, sharey=True)

# fig, axs = plt.subplots(3, 1, figsize=(10, 50))

# fig, axs = plt.subplots(1, 1, figsize=(10, 10), squeeze=False)

for stim_number_i, stim_number in enumerate(stim_numbers[1:]):


	data_ = data.loc[data[us] == stim_number, abs_time]

	us_beg_ = data_.iat[0]

	us_end_ = data_.iat[-1]


	# print(stim_number_i, us_end_ - us_beg_)


	data_plot = data.loc[data[abs_time].between(us_beg_-x0, us_end_+x1)]


	axs[stim_number_i].plot(data_plot[abs_time].to_numpy() - us_beg_, data_plot['Image mean'].to_numpy(), 'k.')
	# axs[stim_number_i].plot(data_plot[abs_time].to_numpy() - us_beg_, data_plot['Angle (deg) 14'].diff().to_numpy(), 'k.')
	# axs[stim_number_i].plot(data_plot[abs_time].to_numpy() - us_beg_, data_plot[galvo_value].to_numpy() + 100, 'bo')
	#! axs[stim_number_i].plot(data__[abs_time].to_numpy() - us_beg_, data__[galvo_value].to_numpy(), 'm')

	axs[stim_number_i].axvline(x=us_beg_ - us_beg_, color='g', linestyle='--')
	axs[stim_number_i].axvline(x=us_end_ - us_beg_, color='r', linestyle='--')
	axs[stim_number_i].set_title(f"Stimulus Number: {stim_number}")
	# plt.plot(time, data__[galvo_value]], 'k')
	# plt.plot(time, data__['Frame beg'], 'bo')
	# plt.plot(time, data__[us], 'yo')
	# plt.plot(time, data__[us_end], 'mo')

	# data__.plot(x=abs_time, y=['Frame beg', 'Image mean', us, us_end], ls='.')

	# break
fig.show()


del data_, data_plot



#endregion

## Read the behavior

In [ ]:
# data.loc[data['Frame beg'].notna()]

data.drop(columns=['Image mean', 'GalvoValue'], inplace=True)





#!!!!!!!!!!!! read the behavior here


#! reverse data_cols to what we want
#! #* Open tail tracking data.
data_cols = x_cols + y_cols + angle_cols
behavior = read_tail_tracking_data(tracking_path).astype('float32')

# behavior.dtypes
# if (tail := read_tail_tracking_data(data_path)) is None: # type: ignore
# 	return None

# plot_behavior_overview(tail, fish_name, fig_behavior_name)

# #* Look for possible tail tracking errors.
# if tracking_errors(tail, single_point_tracking_error_thr):
# 	return None




# Cannot use pd.merge_ordered because of NANs
# data = pd.merge_ordered(data, behavior, on=frame_id, how='left')
all_data = data.copy()
data = all_data.copy()

data = pd.merge(data, behavior, on=frame_id, how='left')

data.reset_index(drop=True, inplace=True)
data.rename(columns={abs_time : 'Time (ms)'}, inplace=True)
abs_time = 'Time (ms)'
data[abs_time] -= data[abs_time].iat[0]

## Organize data in trials

In [ ]:
protocol = data[[abs_time, cs, us]].copy()
protocol = protocol[((protocol[cs]!=0) | (protocol[us]!=0))]

behavior = data[[abs_time] + data_cols].dropna().rename(columns={frame_id : 'Frame number (behavior)'}).copy()
# behavior_array = xr.DataArray(behavior, coords=[('time', all_data.loc[all_data['Frame number'].notna(), :].index), ('parameters', behavior.columns)], dims=['time', 'parameters'])
# behavior_array.to_dataframe()

imaging = xr.DataArray(images, coords={'index': ('time', data.loc[data['Frame beg'].notna(), :].index), 'time': data.loc[data['Frame beg'].notna(), abs_time].to_numpy(), 'x': range(images.shape[1]), 'y': range(images.shape[2])}, dims=['time', 'x', 'y'])

imaging.name = 'Imaging data'
#region Segment data of the different planes.

cs_onset_index = np.array([protocol.loc[protocol[cs] == relevant_cs[i], :].index[0] for i in range(len(relevant_cs))])

index_list = [[i, i+1, i+2*number_imaged_planes, i+2*number_imaged_planes+1] for i in range(0, number_reps_plane_consective * number_imaged_planes, 2)]

planes_cs_onset_indices = cs_onset_index

planes_cs_onset_indices = [cs_onset_index[[j for j in i]] for i in index_list]

del cs_onset_index, index_list


planes_list = []

i = 0
for plane_i, plane_cs_onset_indices in tqdm(enumerate(planes_cs_onset_indices)):

	trials_list = []
	for trial_i, trial_cs_onset_index in enumerate(plane_cs_onset_indices):

		time_start = protocol.loc[trial_cs_onset_index, abs_time] - 45000
		time_end = protocol.loc[trial_cs_onset_index, abs_time] + 15000

		# index = protocol[protocol[abs_time].between(time_start, time_end)].index

		trial_images = imaging.loc[time_start : time_end,:,:]

		# plt.imshow(np.mean(trial_images.values, axis=0))
		# plt.show()


		# images_average = ndimage.median_filter(np.mean(get_good_last_images(trial_images), axis=0), size=median_filter_kernel)
		# # images_average = ndimage.median_filter(np.mean(trial_images[i], axis=0), size=median_filter_kernel)

		# if trial_images[i] is not None:

		# 	planes_numbers[i] = find_plane_in_anatomical_stack(anatomical_stack_images, images_average.astype('float32'), None, x_dim, y_dim)[0]

		# 	#* Bin and filter 3 planes of the anatomical stack where the plane is included.
		# 	reference_images[i] = ndimage.median_filter(np.mean(anatomical_stack_images[planes_numbers[i]-1 : planes_numbers[i]+2], axis=0), size=median_filter_kernel)
		# 	# anatomical_stack_images_sub = ndimage.median_filter(anatomical_stack_images[planes_numbers[i]], size=median_filter_kernel)

		# 	# plt.imshow(reference_images[i])
		# 	# plt.show()

		# else:
		# 	print('Look here')
		# 	planes_numbers[i] = np.nan
		# 	continue


		trials_list.append(Trial(i, protocol[protocol[abs_time].between(time_start, time_end)], behavior[behavior[abs_time].between(time_start, time_end)], trial_images))

		i += 1

	planes_list.append(Plane(trials_list))

	# break

del trials_list

for i in range(len(planes_list)):
	plt.imshow(np.mean(planes_list[i].trials[0].images.values, axis=0))
	plt.show()

## Analysis of the imaging data

### Correct for motion

In [ ]:
#* Read the anatomical stack.
anatomical_stack_images = tifffile.imread(anatomy_1_path).astype('float32')
# anatomical_stack_images = tifffile.imread(anatomy_1_filtered_path).astype('float32')

anatomical_stack_images = ndimage.median_filter(anatomical_stack_images, size=median_filter_kernel, axes=(1,2))


# anatomical_stack_images.shape


#ToDo this should involve the pixel spacing!!!


_, y_dim, x_dim = np.array(anatomical_stack_images.shape)

x_dim = int(x_dim * xy_movement_allowed/2)
y_dim = int(y_dim * xy_movement_allowed/2)


#region Correct motion

for plane_i, plane in tqdm(enumerate(planes_list)):
	# break

	print('Plane: ', plane_i)

	motions = [_ for _ in range(4)]
	template_images = np.zeros((len(plane.trials), plane.trials[0].images.shape[1], plane.trials[0].images.shape[2]))
	plane_numbers = np.zeros(len(plane.trials), dtype='int32')

	#* Motion correction within trial.
	for trial_i, trial in enumerate(plane.trials):

		#* 1.1. Motion correction relative to trials average.

		##* Discard bad frames due to motion, gating of the PMT or plane change when making a template image for the trial.
		# plane.trials[trial_i].images.values, plane.trials[trial_i].template_image, plane.trials[trial_i].position_anatomical_stack 
		motions[trial_i], template_images[trial_i], plane_numbers[trial_i] = correct_motion_within_trial(trial, 5)

		plt.imshow(anatomical_stack_images[plane_numbers[trial_i]])
		plt.show()


		#* Frames to ignore due to too much motion (or gating of the PMT, which causes a huge "motion").
		trial_images = trial.images.values

		# Mask with True where the frames are bad (due to gating of the PMT or motion).
		mask_bad_frames = (~get_good_images_indices(trial_images)) | (np.where(get_total_motion(motions[trial_i]) > motion_thr_from_trial_average, True, False))

		planes_list[plane_i].trials[trial_i].mask_bad_frames = mask_bad_frames


	#* Motion correction across trials of the same plane.
	for trial_i, trial in enumerate(plane.trials):
		
		if trial_i > 0:
			#* Measure motion of each frame using phase cross-correlation.
			motion = measure_motion(np.expand_dims(template_images[trial_i][5:-5, 5:-5], axis=0), template_images[0][5:-5, 5:-5], normalization=None)[0]

			motions[trial_i] += motion

		#* Measure motion of each frame using phase cross-correlation.
		total_motion = get_total_motion(motions[trial_i])
		# Use half of the frames to get the template image.
		motion_thr = np.median(total_motion)

		
		
		#* Align the frames to their average.
		aligned_frames = align_frames(trial.images.to_numpy(), motions[trial_i], total_motion, [5,10,30,35][trial_i])

		template_image = get_template_image(aligned_frames[np.where(total_motion <= motion_thr)[0]])


		#* Identify the plane number of the trial.
		plane_number, _ = find_plane_in_anatomical_stack(anatomical_stack_images, template_image.astype('float32'), None, x_dim, y_dim)


		plt.imshow(ndimage.median_filter(np.mean(aligned_frames, axis=0), size=median_filter_kernel))
		plt.colorbar(shrink=0.5)
		plt.show()

		planes_list[plane_i].trials[trial_i].images.values = aligned_frames
		planes_list[plane_i].trials[trial_i].template_image = template_image
		planes_list[plane_i].trials[trial_i].position_anatomical_stack = plane_number

		# break
	break
	print('Plane:', plane_i, plane_numbers)

#endregion

### Save the data

In [ ]:
#* Save the data.
path_pkl = path_home / fish_name / (fish_name + '.pkl')


with open(path_pkl, 'wb') as f:
	pickle.dump(planes_list, f)

### Read the data

In [13]:
#* Save the data.
path_pkl = path_home / fish_name / (fish_name + '.pkl')

with open(path_pkl, 'rb') as f:
	planes_list_new = pickle.load(f)